# 7.5 DBSCAN

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 7: Clustering Techniques

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-07-clustering-techniques/07.05_dbscan.ipynb)

Companion code for Section 7.5 of the book. Run the cells in order from top to bottom.

**Example 7.5.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_blobs, make_circles
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
import warnings; warnings.filterwarnings('ignore')

# ── DBSCAN on different cluster shapes ────────────────────────────
np.random.seed(42)

# Generate 3 different datasets
X_moons,  y_moons  = make_moons(n_samples=400, noise=0.08, random_state=42)
X_blobs,  y_blobs  = make_blobs(n_samples=400, centers=4,  random_state=42)
X_circles,y_circles= make_circles(n_samples=400,noise=0.05,factor=0.5,random_state=42)

datasets = [
    ('Two Moons (non-convex)', X_moons,   y_moons,   0.15, 5),
    ('4 Blobs (spherical)',    X_blobs,   y_blobs,   0.80, 5),
    ('Concentric Circles',    X_circles, y_circles, 0.15, 5),
]

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('DBSCAN - True Labels vs K-Means vs DBSCAN', fontsize=14, fontweight='bold')

for row, (name, X_data, y_data, eps, min_pts) in enumerate(datasets):
    X_sc = StandardScaler().fit_transform(X_data)

    # True labels
    ax = axes[row, 0]
    for k in np.unique(y_data):
        ax.scatter(X_sc[y_data==k,0], X_sc[y_data==k,1], s=15, alpha=0.7)
    ax.set_title(f'{name}\nTrue Labels'); ax.grid(alpha=0.3)

    # K-Means
    from sklearn.cluster import KMeans
    k_val = len(np.unique(y_data))
    km = KMeans(n_clusters=k_val, random_state=42, n_init=10)
    km_labels = km.fit_predict(X_sc)
    ari_km = adjusted_rand_score(y_data, km_labels)
    ax = axes[row, 1]
    for k in np.unique(km_labels):
        ax.scatter(X_sc[km_labels==k,0], X_sc[km_labels==k,1], s=15, alpha=0.7)
    ax.set_title(f'K-Means (k={k_val})\nARI={ari_km:.3f}'); ax.grid(alpha=0.3)

    # DBSCAN
    db = DBSCAN(eps=eps, min_samples=min_pts)
    db_labels = db.fit_predict(X_sc)
    n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise    = (db_labels == -1).sum()
    ari_db = adjusted_rand_score(y_data, db_labels)
    ax = axes[row, 2]
    unique_labels = set(db_labels)
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
    for color, k in zip(colors, sorted(unique_labels)):
        if k == -1:
            ax.scatter(X_sc[db_labels==k,0], X_sc[db_labels==k,1],
                      s=40, c='black', marker='x', alpha=0.8, label='Noise')
        else:
            ax.scatter(X_sc[db_labels==k,0], X_sc[db_labels==k,1],
                      s=15, alpha=0.7, color=color)
    ax.set_title(f'DBSCAN (ε={eps}, minPts={min_pts})\n'
               f'{n_clusters} clusters, {n_noise} noise pts, ARI={ari_db:.3f}')
    ax.grid(alpha=0.3)
    if n_noise > 0: ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('dbscan_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── How to choose ε: k-distance plot ─────────────────────────────
print('\nChoosing ε - k-distance plot method:')
X_sc = StandardScaler().fit_transform(X_moons)
nbrs = NearestNeighbors(n_neighbors=5).fit(X_sc)
distances, _ = nbrs.kneighbors(X_sc)
k_distances = np.sort(distances[:, -1])[::-1]   # Distance to 5th neighbour

# The 'knee' of this curve is the optimal ε
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_distances, 'b-', lw=2)
ax.axhline(y=0.15, color='red', ls='--', lw=2, label='Suggested ε=0.15 (knee point)')
ax.set_xlabel('Points sorted by 5th-NN distance'); ax.set_ylabel('5th-NN Distance')
ax.set_title('k-Distance Plot for ε Selection\n(Look for the "elbow" - steep drop)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('dbscan_epsilon_selection.png', dpi=150, bbox_inches='tight')
plt.show()

# ── DBSCAN parameter sensitivity ─────────────────────────────────
print('\nDBSCAN parameter sensitivity:')
print(f'{"eps":<8} {"minPts":<10} {"Clusters":>10} {"Noise":>8} {"ARI":>8}')
print('='*50)
for eps_val in [0.05, 0.10, 0.15, 0.20, 0.30]:
    for min_pts in [3, 5, 10]:
        db = DBSCAN(eps=eps_val, min_samples=min_pts)
        lbl = db.fit_predict(X_sc)
        nc = len(set(lbl)) - (1 if -1 in lbl else 0)
        nn = (lbl==-1).sum()
        ari = adjusted_rand_score(y_moons, lbl) if nc > 1 else 0
        if min_pts == 5:
            print(f'{eps_val:<8.2f} {min_pts:<10} {nc:>10} {nn:>8} {ari:>8.4f}')